# Missão Aurora Siger — Verificação de Pré-Decolagem

**Atividade Integradora — Análise de telemetria e decisão de lançamento**

Este notebook lê a telemetria simulada da nave *Aurora Siger*, compara cada leitura com as
faixas seguras definidas para a missão e conclui se o lançamento está liberado
(`PRONTO PARA DECOLAR`) ou se deve ser interrompido (`DECOLAGEM ABORTADA`).
Ao final, calcula a autonomia energética inicial da nave após o lançamento.

**Como executar:** rode as células na ordem, de cima para baixo (`Shift + Enter`).
Não é necessário instalar nenhuma biblioteca — o notebook usa apenas Python puro.

---
## 1. Organização e descrição da telemetria

A nave envia cinco grupos de leituras. Cada um tem um papel diferente na decisão:

| Parâmetro | Unidade | O que representa | Faixa segura |
|---|---|---|---|
| Temperatura interna | °C | Temperatura da cabine e dos compartimentos pressurizados | 18 a 26 °C |
| Temperatura externa | °C | Temperatura do ambiente ao redor do casco | Somente registro |
| Integridade estrutural | 0 ou 1 | 1 = casco íntegro, 0 = falha detectada | Precisa ser 1 |
| Nível de energia | % | Carga restante das baterias principais | Acima de 80% |
| Pressão dos tanques | atm | Pressão interna dos tanques de propelente | 30 a 50 atm |
| Módulos críticos | texto | Navegação, suporte à vida e comunicação | Precisa ser "OK" |

A temperatura externa é registrada no relatório, mas não entra na decisão: a nave é projetada
para operar em ambientes extremos, e o que importa para a tripulação é a temperatura **interna**.
Já a integridade estrutural é binária de propósito — não existe "meio íntegro" em um casco que
vai ser submetido à pressão do lançamento.

In [1]:
# ---------------------------------------------------------------
# 1. LEITURA DOS DADOS (telemetria simulada)
# ---------------------------------------------------------------
telemetria = {
    "temperatura_interna": 22.5,     # °C - cabine
    "temperatura_externa": -45.0,    # °C - apenas registro, não impede o lançamento
    "integridade_estrutural": 1,     # 1 = casco íntegro / 0 = falha detectada
    "nivel_energia": 85.0,           # % da carga das baterias
    "pressao_tanques": 42.0,         # atm
    "modulos_criticos": "OK",        # OK = todos os sistemas essenciais operantes
}

# ---------------------------------------------------------------
# 2. FAIXAS SEGURAS DEFINIDAS PARA A MISSÃO
# ---------------------------------------------------------------
TEMP_INTERNA_MIN = 18.0
TEMP_INTERNA_MAX = 26.0
ENERGIA_MINIMA = 80.0
PRESSAO_MIN = 30.0
PRESSAO_MAX = 50.0

for chave, valor in telemetria.items():
    print(f"{chave:<24}: {valor}")

temperatura_interna     : 22.5
temperatura_externa     : -45.0
integridade_estrutural  : 1
nivel_energia           : 85.0
pressao_tanques         : 42.0
modulos_criticos        : OK


---
## 2. Algoritmo de verificação

### Pseudocódigo

```
INÍCIO
    LER temperatura_interna, temperatura_externa, integridade_estrutural,
        nivel_energia, pressao_tanques, modulos_criticos

    REGISTRAR temperatura_externa no log da missão   // informativo

    SE (temperatura_interna ENTRE 18 E 26)
       E (integridade_estrutural = 1)
       E (nivel_energia > 80)
       E (pressao_tanques ENTRE 30 E 50)
       E (modulos_criticos = "OK")
    ENTÃO
        ESCREVER "PRONTO PARA DECOLAR"
    SENÃO
        ESCREVER "DECOLAGEM ABORTADA"
    FIM SE
FIM
```

A regra é conservadora de propósito: basta **um** parâmetro fora da faixa para abortar.
Em um lançamento não existe compensação — energia sobrando não cobre um casco comprometido.

### Fluxograma

![Fluxograma de decisão](docs/fluxograma_decisao.png)

In [2]:
# ---------------------------------------------------------------
# 3. EXECUÇÃO DAS VERIFICAÇÕES
# ---------------------------------------------------------------
def verificar_telemetria(t):
    """Compara cada leitura com sua faixa segura.
    Retorna o status final e a lista detalhada das checagens."""

    checagens = [
        ("Temperatura interna",
         TEMP_INTERNA_MIN <= t["temperatura_interna"] <= TEMP_INTERNA_MAX,
         f'{t["temperatura_interna"]} °C (faixa: {TEMP_INTERNA_MIN} a {TEMP_INTERNA_MAX} °C)'),

        ("Integridade estrutural",
         t["integridade_estrutural"] == 1,
         f'{t["integridade_estrutural"]} (esperado: 1)'),

        ("Nível de energia",
         t["nivel_energia"] > ENERGIA_MINIMA,
         f'{t["nivel_energia"]}% (mínimo: acima de {ENERGIA_MINIMA}%)'),

        ("Pressão dos tanques",
         PRESSAO_MIN <= t["pressao_tanques"] <= PRESSAO_MAX,
         f'{t["pressao_tanques"]} atm (faixa: {PRESSAO_MIN} a {PRESSAO_MAX} atm)'),

        ("Módulos críticos",
         t["modulos_criticos"] == "OK",
         f'{t["modulos_criticos"]} (esperado: OK)'),
    ]

    aprovado_em_tudo = all(ok for _, ok, _ in checagens)
    status = "PRONTO PARA DECOLAR" if aprovado_em_tudo else "DECOLAGEM ABORTADA"
    return status, checagens


def exibir_relatorio(t):
    print("--- INICIANDO VERIFICAÇÃO DE TELEMETRIA ---\n")

    status, checagens = verificar_telemetria(t)

    for nome, aprovado, leitura in checagens:
        selo = "APROVADO" if aprovado else "REPROVADO"
        print(f"[{selo}] {nome:<24} -> {leitura}")

    print(f'[REGISTRO] {"Temperatura externa":<24} -> {t["temperatura_externa"]} °C '
          "(informativo, não bloqueia o lançamento)")

    print(f"\nSTATUS FINAL: {status}")
    return status


# ---------------------------------------------------------------
# 4. RESULTADO FINAL IMPRESSO
# ---------------------------------------------------------------
status_final = exibir_relatorio(telemetria)

--- INICIANDO VERIFICAÇÃO DE TELEMETRIA ---

[APROVADO] Temperatura interna      -> 22.5 °C (faixa: 18.0 a 26.0 °C)
[APROVADO] Integridade estrutural   -> 1 (esperado: 1)
[APROVADO] Nível de energia         -> 85.0% (mínimo: acima de 80.0%)
[APROVADO] Pressão dos tanques      -> 42.0 atm (faixa: 30.0 a 50.0 atm)
[APROVADO] Módulos críticos         -> OK (esperado: OK)
[REGISTRO] Temperatura externa      -> -45.0 °C (informativo, não bloqueia o lançamento)

STATUS FINAL: PRONTO PARA DECOLAR


### 2.1 Implementação direta da condição

A célula acima quebra a verificação item a item para deixar claro onde cada leitura passou ou
falhou. A decisão em si, porém, cabe em uma única condição composta — que é exatamente a
tradução do pseudocódigo para Python:

In [3]:
# Mesma lógica, na forma condensada
temperatura_interna = telemetria["temperatura_interna"]
temperatura_externa = telemetria["temperatura_externa"]
integridade_estrutural = telemetria["integridade_estrutural"]
nivel_energia = telemetria["nivel_energia"]
pressao_tanques = telemetria["pressao_tanques"]
modulos_criticos = telemetria["modulos_criticos"]

if (18 <= temperatura_interna <= 26) and \
   (integridade_estrutural == 1) and \
   (nivel_energia > 80) and \
   (30 <= pressao_tanques <= 50) and \
   (modulos_criticos == "OK"):

    print("STATUS FINAL: PRONTO PARA DECOLAR")
else:
    print("STATUS FINAL: DECOLAGEM ABORTADA")

STATUS FINAL: PRONTO PARA DECOLAR


---
## 3. Análise energética

A autonomia inicial é o que sobra nas baterias depois que a nave rompe a atmosfera.
O cálculo parte da carga atual, desconta as perdas térmicas de transmissão e conversão
e depois o consumo do próprio lançamento:

```
Carga atual        = capacidade total × (nível de energia / 100)
Perdas térmicas    = carga atual × 5%
Consumo total      = consumo da decolagem + perdas térmicas
Autonomia inicial  = carga atual − consumo total
```

In [4]:
# ---------------------------------------------------------------
# 5. ANÁLISE ENERGÉTICA
# ---------------------------------------------------------------
CAPACIDADE_TOTAL = 100_000.0   # kWh
CONSUMO_DECOLAGEM = 30_000.0   # kWh
TAXA_PERDAS = 0.05             # 5% de dissipação térmica

carga_atual = CAPACIDADE_TOTAL * (telemetria["nivel_energia"] / 100)
perdas = carga_atual * TAXA_PERDAS
consumo_total = CONSUMO_DECOLAGEM + perdas
autonomia_inicial = carga_atual - consumo_total

print("--- ANÁLISE ENERGÉTICA ---")
print(f"Capacidade total ............: {CAPACIDADE_TOTAL:,.2f} kWh")
print(f'Carga atual ({telemetria["nivel_energia"]}%) .........: {carga_atual:,.2f} kWh')
print(f"Perdas térmicas (5%) ........: {perdas:,.2f} kWh")
print(f"Consumo na decolagem ........: {CONSUMO_DECOLAGEM:,.2f} kWh")
print(f"Consumo total de lançamento .: {consumo_total:,.2f} kWh")
print(f"AUTONOMIA INICIAL ...........: {autonomia_inicial:,.2f} kWh")
print(f"Reserva sobre a carga atual .: {autonomia_inicial / carga_atual * 100:.2f}%")

--- ANÁLISE ENERGÉTICA ---
Capacidade total ............: 100,000.00 kWh
Carga atual (85.0%) .........: 85,000.00 kWh
Perdas térmicas (5%) ........: 4,250.00 kWh
Consumo na decolagem ........: 30,000.00 kWh
Consumo total de lançamento .: 34,250.00 kWh
AUTONOMIA INICIAL ...........: 50,750.00 kWh
Reserva sobre a carga atual .: 59.71%


**Leitura do resultado:** a nave chega à órbita com **50.750 kWh**, pouco mais da metade da
carga com que iniciou a contagem. É uma margem confortável para manter suporte à vida,
navegação e comunicação até a primeira janela de recarga solar.

---
## 4. Teste do caminho de aborto

Um algoritmo de segurança só vale se ele realmente barra o que precisa barrar. A célula abaixo
repete a verificação com uma telemetria comprometida — casco com falha, bateria abaixo do
mínimo e tanque sobrepressurizado — para confirmar que o `else` dispara.

In [5]:
# ---------------------------------------------------------------
# 6. CENÁRIO ANÔMALO (validação do caminho de aborto)
# ---------------------------------------------------------------
telemetria_falha = {
    "temperatura_interna": 31.0,     # acima do limite de 26 °C
    "temperatura_externa": -45.0,
    "integridade_estrutural": 0,     # falha no casco
    "nivel_energia": 62.0,           # abaixo dos 80% exigidos
    "pressao_tanques": 57.0,         # acima das 50 atm
    "modulos_criticos": "FALHA",
}

status_falha = exibir_relatorio(telemetria_falha)

--- INICIANDO VERIFICAÇÃO DE TELEMETRIA ---

[REPROVADO] Temperatura interna      -> 31.0 °C (faixa: 18.0 a 26.0 °C)
[REPROVADO] Integridade estrutural   -> 0 (esperado: 1)
[REPROVADO] Nível de energia         -> 62.0% (mínimo: acima de 80.0%)
[REPROVADO] Pressão dos tanques      -> 57.0 atm (faixa: 30.0 a 50.0 atm)
[REPROVADO] Módulos críticos         -> FALHA (esperado: OK)
[REGISTRO] Temperatura externa      -> -45.0 °C (informativo, não bloqueia o lançamento)

STATUS FINAL: DECOLAGEM ABORTADA


---
## 5. Conclusão

Com a telemetria nominal, os cinco parâmetros críticos ficaram dentro das faixas seguras e o
sistema liberou o lançamento, com autonomia inicial de 50.750 kWh. No cenário anômalo, as
cinco checagens reprovaram e a decolagem foi abortada, como esperado.

A análise completa — classificação dos dados, anomalias, avaliação de risco e a reflexão
crítica sobre ética, impacto social e sustentabilidade — está no relatório em PDF que
acompanha este repositório.